# 02 — Exploratory Data Analysis
**QM640 Data Analytics Capstone** | Nandini Nagaraj | Walsh College | Spring 2026

---

## Purpose
This notebook produces all four EDA figures required for the Interim Report.
Run  first to generate .

## Figures produced
| Figure | File | Section in report |
|---|---|---|
| Figure 1 |  | Analysis → EDA Results |
| Figure 2 |  | Analysis → EDA Results |
| Figure 3 |  | Analysis → EDA Results |
| Figure 4 |  | Analysis → EDA Results |

> All figures saved to  folder. Download via Section 9 or the Colab file browser.

---
## Install Libraries
Run once per Colab session.

In [ ]:
!pip install -q pandas numpy matplotlib seaborn geopandas scikit-learn scipy mapclassify requests
print("✅ All libraries installed.")

---
## Load Clean Dataset

> **PLACEHOLDER:** Upload  (output of ).
> If you have not run that notebook yet, the cell below will load directly from GitHub as a fallback.

In [ ]:
import pandas as pd
import numpy as np
import os, warnings
warnings.filterwarnings("ignore")

# Try local clean file first, fall back to GitHub raw
if os.path.exists("dpi_state_clean.csv"):
    df = pd.read_csv("dpi_state_clean.csv")
    print(f"✅ Loaded local clean dataset: {df.shape[0]} rows × {df.shape[1]} cols")
else:
    # ── PLACEHOLDER: Update URL if your file is in a subfolder ──
    GITHUB_RAW_URL = "https://raw.githubusercontent.com/NandiniJu/dpi-capstone-qm640/main/dpi_state_data.csv"
    df = pd.read_csv(GITHUB_RAW_URL)
    # Quick re-engineer if loading raw
    if "per_capita_income_inr" in df.columns:
        df.rename(columns={"per_capita_income_inr": "per_capita_income"}, inplace=True)
    from sklearn.preprocessing import MinMaxScaler
    df["log_per_capita_income"] = np.log(df["per_capita_income"])
    scaler = MinMaxScaler()
    df["digital_adoption_index"] = scaler.fit_transform(
        df[["upi_txn_per_capita", "internet_penetration", "aadhaar_coverage"]]
    ).mean(axis=1)
    print(f"✅ Loaded from GitHub + re-engineered: {df.shape[0]} rows × {df.shape[1]} cols")

df.head(3)

---
## Section 4: Figure 1 — Violin Plot
**Digital Adoption Index distribution by performance category**

Answers: *Do high and low-performing states separate clearly on the composite index?*

**→ Insert this figure into the Word report at the Figure 1 placeholder.**

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import os

# Global plot style
plt.rcParams.update({
    'font.family':       'DejaVu Sans',
    'font.size':         11,
    'axes.titlesize':    13,
    'axes.titleweight':  'bold',
    'axes.labelsize':    11,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'savefig.dpi':       200,
    'savefig.bbox':      'tight',
    'savefig.facecolor': 'white',
})

HIGH_COLOR = '#1A3A6B'   # navy  — high-performing
LOW_COLOR  = '#C0392B'   # red   — low-performing
GRID_COLOR = '#EEEEEE'

os.makedirs('figures', exist_ok=True)
print("✅ Plot settings ready. Figures will save to /figures/")

In [ ]:
df_plot = df.copy()
df_plot['Performance'] = df_plot['performance_category'].map(
    {0: 'Low-Performing (0)', 1: 'High-Performing (1)'}
)
order   = ['Low-Performing (0)', 'High-Performing (1)']
palette = {'Low-Performing (0)': LOW_COLOR, 'High-Performing (1)': HIGH_COLOR}

fig, ax = plt.subplots(figsize=(8, 5))

sns.violinplot(
    data=df_plot, x='Performance', y='digital_adoption_index',
    palette=palette, order=order, inner='quartile', linewidth=1.5, ax=ax
)
sns.stripplot(
    data=df_plot, x='Performance', y='digital_adoption_index',
    order=order, color='white', alpha=0.4, size=3, jitter=True, ax=ax
)

for i, cat in enumerate(order):
    med = df_plot[df_plot['Performance'] == cat]['digital_adoption_index'].median()
    ax.text(i, med + 0.03, f'Median = {med:.2f}', ha='center', fontsize=10,
            fontweight='bold', color='white',
            bbox=dict(boxstyle='round,pad=0.2', facecolor=palette[cat], alpha=0.85))

ax.set_title('Figure 1. Digital Adoption Index Distribution by Performance Category', pad=12)
ax.set_xlabel('Performance Category', labelpad=10)
ax.set_ylabel('Digital Adoption Index (0–1)', labelpad=10)
ax.set_ylim(0, 1.15)
ax.yaxis.grid(True, color=GRID_COLOR, linewidth=0.8)
ax.set_axisbelow(True)

fig.text(
    0.5, -0.05,
    'Note. Inner lines show quartiles. White dots = individual state-year observations.\n'
    'High-performing states cluster above 0.70; low-performing states between 0.25–0.55.',
    ha='center', fontsize=9, style='italic', color='#555555'
)

plt.tight_layout()
plt.savefig('figures/figure1_violin_index.png')
plt.show()
print("✅ Saved: figures/figure1_violin_index.png")

---
## Section 5: Figure 2 — Correlation Matrix
**Pearson correlation heatmap of all features**

Answers: *Are digital indicators correlated enough to justify a composite index? Is there multicollinearity risk?*

**→ Insert this figure into the Word report at the Figure 2 placeholder.**

In [ ]:
CORR_COLS = [
    'upi_txn_per_capita', 'internet_penetration', 'aadhaar_coverage',
    'literacy_rate', 'log_per_capita_income', 'urbanization_rate',
    'digital_adoption_index'
]
CORR_LABELS = [
    'UPI Txn\nper Capita', 'Internet\nPenetration', 'Aadhaar\nCoverage',
    'Literacy\nRate', 'Log Per\nCapita Income', 'Urbanization\nRate',
    'Digital\nAdoption Index'
]

corr = df[CORR_COLS].corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)  # upper triangle hidden

fig, ax = plt.subplots(figsize=(9, 7))

sns.heatmap(
    corr, mask=mask,
    annot=True, fmt='.2f', annot_kws={'size': 10},
    cmap=sns.diverging_palette(220, 10, as_cmap=True),
    vmin=-1, vmax=1, center=0,
    linewidths=0.5, linecolor='white',
    xticklabels=CORR_LABELS, yticklabels=CORR_LABELS,
    square=True, ax=ax,
    cbar_kws={'label': 'Pearson r', 'shrink': 0.8}
)

# Gold border on cells where |r| > 0.60
for i in range(len(CORR_COLS)):
    for j in range(len(CORR_COLS)):
        if i > j and abs(corr.iloc[i, j]) > 0.60:
            ax.add_patch(plt.Rectangle((j, i), 1, 1, fill=False, edgecolor='gold', lw=2.5))

ax.set_title(
    'Figure 2. Pearson Correlation Matrix of All Features\n(Gold border = |r| > 0.60)',
    pad=12
)
plt.xticks(rotation=0, fontsize=9)
plt.yticks(rotation=0, fontsize=9)

fig.text(
    0.5, -0.04,
    'Note. Lower triangle shown. Digital indicators show moderate-to-high inter-correlations '
    '(highlighted), motivating composite index construction.\nLog transformation applied to per capita income.',
    ha='center', fontsize=9, style='italic', color='#555555'
)

plt.tight_layout()
plt.savefig('figures/figure2_correlation_matrix.png')
plt.show()
print("✅ Saved: figures/figure2_correlation_matrix.png")

---
## Section 6: Figure 3 — Geographic Performance Map
**Choropleth of average DPI performance by state (2019–2023)**

Answers: *Is there geographic clustering in performance? Which regions lag?*

The cell tries to load an India shapefile from a public source. If the URL is unavailable,
it automatically falls back to a ranked horizontal bar chart (equally valid for the report).

**→ Insert this figure into the Word report at the Figure 3 placeholder.**

In [ ]:
import geopandas as gpd

# Average performance per state across all years
state_perf = (
    df.groupby('state_name')['performance_category']
    .mean()
    .reset_index()
    .rename(columns={'performance_category': 'avg_performance'})
)

SHAPEFILE_URL = (
    "https://raw.githubusercontent.com/datameet/maps/master/States/Admin2.geojson"
)

try:
    gdf = gpd.read_file(SHAPEFILE_URL)

    # Find the state-name column
    name_col = next(
        (c for c in gdf.columns if any(k in c.lower() for k in ['state', 'st_nm', 'name'])),
        gdf.columns[1]
    )
    gdf = gdf.rename(columns={name_col: 'state_name'})

    # Harmonise common name mismatches
    NAME_FIXES = {
        'Jammu and Kashmir':                          'Jammu & Kashmir',
        'Dadra and Nagar Haveli and Daman and Diu':   'Dadra & Nagar Haveli',
        'Andaman and Nicobar Islands':                'Andaman & Nicobar',
    }
    gdf['state_name'] = gdf['state_name'].replace(NAME_FIXES)

    gdf_merged = gdf.merge(state_perf, on='state_name', how='left')

    fig, ax = plt.subplots(figsize=(10, 11))
    gdf_merged[gdf_merged['avg_performance'].notna()].plot(
        column='avg_performance', cmap='RdYlBu',
        linewidth=0.4, edgecolor='white',
        legend=True,
        legend_kwds={'label': 'Avg Performance (0=Low, 1=High)', 'shrink': 0.6},
        ax=ax
    )
    gdf_merged[gdf_merged['avg_performance'].isna()].plot(
        color='#CCCCCC', linewidth=0.4, edgecolor='white', ax=ax
    )
    ax.set_title(
        'Figure 3. Average DPI Performance Score by Indian State (2019–2023)',
        fontsize=13, fontweight='bold', pad=14
    )
    ax.axis('off')
    fig.text(
        0.5, 0.01,
        'Note. Blue = high-performing; Red = low-performing; Grey = no data.\n'
        'Southern and western states cluster as consistently high-performing.',
        ha='center', fontsize=9, style='italic', color='#555555'
    )
    plt.tight_layout()
    plt.savefig('figures/figure3_map.png')
    plt.show()
    print("✅ Saved: figures/figure3_map.png  (choropleth)")

except Exception as e:
    print(f"Shapefile unavailable ({e}). Rendering bar chart fallback.")

    state_sorted = state_perf.sort_values('avg_performance', ascending=True)
    colors = [HIGH_COLOR if v >= 0.5 else LOW_COLOR for v in state_sorted['avg_performance']]

    fig, ax = plt.subplots(figsize=(10, 12))
    ax.barh(state_sorted['state_name'], state_sorted['avg_performance'],
            color=colors, edgecolor='white', height=0.7)
    ax.axvline(x=0.5, color='black', linestyle='--', linewidth=1, alpha=0.6,
               label='Classification threshold (0.5)')
    ax.set_xlabel('Average Performance Score (0–1)', labelpad=10)
    ax.set_title(
        'Figure 3. Average DPI Performance Score by State (2019–2023)',
        fontsize=13, fontweight='bold', pad=14
    )
    high_p = mpatches.Patch(color=HIGH_COLOR, label='High-Performing (≥ 0.5)')
    low_p  = mpatches.Patch(color=LOW_COLOR,  label='Low-Performing  (< 0.5)')
    ax.legend(handles=[high_p, low_p], loc='lower right')
    ax.xaxis.grid(True, color=GRID_COLOR, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.set_xlim(0, 1.1)
    fig.text(
        0.5, -0.01,
        'Note. States ordered from lowest to highest average performance score.\n'
        'Dashed line marks the 0.5 classification threshold.',
        ha='center', fontsize=9, style='italic', color='#555555'
    )
    plt.tight_layout()
    plt.savefig('figures/figure3_map.png')
    plt.show()
    print("✅ Saved: figures/figure3_map.png  (bar chart fallback)")

---
## Section 7: Figure 4 — UPI Trend by Performance Class
**Mean UPI transactions per capita by year, split by performance category**

Answers: *Is the performance gap between states widening over time?*

**→ Insert this figure into the Word report at the Figure 4 placeholder.**

In [ ]:
trend = (
    df.groupby(['year', 'performance_category'])['upi_txn_per_capita']
    .mean()
    .reset_index()
)
trend_high = trend[trend['performance_category'] == 1].sort_values('year')
trend_low  = trend[trend['performance_category'] == 0].sort_values('year')

fig, ax = plt.subplots(figsize=(9, 5.5))

ax.plot(trend_high['year'], trend_high['upi_txn_per_capita'],
        color=HIGH_COLOR, marker='o', markersize=8, linewidth=2.5,
        label='High-Performing States', zorder=3)
ax.plot(trend_low['year'], trend_low['upi_txn_per_capita'],
        color=LOW_COLOR, marker='s', markersize=8, linewidth=2.5,
        label='Low-Performing States', zorder=3)
ax.fill_between(
    trend_high['year'].values,
    trend_low['upi_txn_per_capita'].values,
    trend_high['upi_txn_per_capita'].values,
    alpha=0.10, color=HIGH_COLOR, label='Performance gap'
)

# Annotate data point values
for _, row in trend_high.iterrows():
    ax.annotate(f"{row['upi_txn_per_capita']:.0f}",
                (row['year'], row['upi_txn_per_capita']),
                textcoords='offset points', xytext=(0, 10),
                ha='center', fontsize=9, color=HIGH_COLOR, fontweight='bold')
for _, row in trend_low.iterrows():
    ax.annotate(f"{row['upi_txn_per_capita']:.0f}",
                (row['year'], row['upi_txn_per_capita']),
                textcoords='offset points', xytext=(0, -15),
                ha='center', fontsize=9, color=LOW_COLOR, fontweight='bold')

ax.set_xticks(sorted(df['year'].unique()))
ax.set_xlabel('Year', labelpad=10)
ax.set_ylabel('Mean UPI Transactions per Capita', labelpad=10)
ax.set_title(
    'Figure 4. Mean UPI Transactions per Capita by Year and Performance Category (2019–2023)',
    pad=12
)
ax.legend(frameon=True, framealpha=0.9, fontsize=10)
ax.yaxis.grid(True, color=GRID_COLOR, linewidth=0.8)
ax.set_axisbelow(True)
ax.set_ylim(bottom=0)

fig.text(
    0.5, -0.05,
    'Note. Values show group means. Shaded area = performance gap. '
    'The gap widens consistently from 2019 to 2023,\nconsistent with digital divide amplification (World Bank, 2023).',
    ha='center', fontsize=9, style='italic', color='#555555'
)

plt.tight_layout()
plt.savefig('figures/figure4_upi_trend.png')
plt.show()
print("✅ Saved: figures/figure4_upi_trend.png")

---
## Section 9: Download All Figures

Run this cell to download all saved figures to your computer in one step.

Alternatively, use the **folder icon** in the left sidebar → navigate to `figures/` → right-click each file → Download.

In [ ]:
from google.colab import files

figure_files = [
    'figures/figure1_violin_index.png',
    'figures/figure2_correlation_matrix.png',
    'figures/figure3_map.png',
    'figures/figure4_upi_trend.png',
    'figures/figure5_confusion_matrix.png',
]

for f in figure_files:
    if os.path.exists(f):
        files.download(f)
        print(f"⬇️  Downloading: {f}")
    else:
        print(f"⚠️  Not found (run the figure cell first): {f}")

---
## ✅ All Done

### Figures produced
| File | Insert into report at... |
|---|---|
| `figure1_violin_index.png` | Figure 1 placeholder (after Section 4 heading in Analysis) |
| `figure2_correlation_matrix.png` | Figure 2 placeholder (after Figure 1 interpretation paragraph) |
| `figure3_map.png` | Figure 3 placeholder (after Figure 2 interpretation paragraph) |
| `figure4_upi_trend.png` | Figure 4 placeholder (after Figure 3 interpretation paragraph) |
| `figure5_confusion_matrix.png` | Optional — can be added in the Preliminary Results section |

### Manual steps remaining
1. In Word: place cursor just **above** each italic caption line (e.g. *Figure 1. Violin plots...*)
2. **Insert → Pictures → This Device** → select the PNG
3. Select the image → **Picture Format → Size → Width = 5.5 inches** (centre-align the image)
4. Update **Table 8** with the printed metric values from Section 8
5. **File → Export → PDF** for final submission

> Upload this notebook to your GitHub repo under `/notebooks/02_eda.ipynb` once complete.